In [1]:
import sys

sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import image, plotting
import lib.mni_to_atlas as mni_to_atlas
from nilearn.reporting import get_clusters_table
from glob import glob
import nibabel

mni_to_atlas._ATLASES_PATH = "../lib"

atlas = mni_to_atlas.AtlasBrowser("AAL3")


In [2]:

images_files = glob("../contrasts/group_p_values/*.nii.gz")

contrast_types = {v.split("\\")[-1].split(".nii.gz")[0] for v in images_files}

contrast_types

{'25_+_35_over_5_+_15',
 '35_+_45_over_5_+_15',
 '45_+_55_over_5_+_15',
 '55_+_65_over_5_+_15',
 '65_+_75_over_5_+_15',
 '75_+_85_over_5_+_15',
 '85_+_95_over_5_+_15',
 'high_over_low',
 'undecided_over_high_+_low'}

In [3]:
images = {
    contrast_type: nibabel.load(f"../contrasts/group_p_values/{contrast_type}.nii.gz")
    for contrast_type in contrast_types
}

null_distribs = {
    contrast_type: np.load(f"../contrasts/group_p_values/{contrast_type}_null.npy")
    for contrast_type in contrast_types
}


In [4]:
found_regions = {}
thresholds = [0.05, 0.01]

for contrast_type in contrast_types:
    image = images[contrast_type]
    null = null_distribs[contrast_type]

    found_regions[contrast_type] = {}

    for threshold in thresholds:
        percentile = np.abs((1 - threshold) * 100)
        stat_threshold = np.percentile(null, percentile)

        found_regions[contrast_type][threshold] = []

        table = get_clusters_table(
            image, stat_threshold=stat_threshold, cluster_threshold=1
        )

        pos = [np.array([x,y,z]) for (x,y,z) in zip(table['X'], table['Y'], table['Z'])]

        for p in pos:
            projected_coords = atlas.project_to_nearest(p)
            projected_regions = atlas.find_regions(projected_coords)

            found_regions[contrast_type][threshold].append(*projected_regions)


C:\Users\ducat\AppData\Local\Temp\ipykernel_17820\2850162604.py:16: UserWarning: The given float value must not exceed 0.3902989066702947. But, you have given threshold=5.037824947424305.
  table = get_clusters_table(
C:\Users\ducat\AppData\Local\Temp\ipykernel_17820\2850162604.py:16: UserWarning: No clusters found with stat higher than 5.037824947424305
  table = get_clusters_table(
C:\Users\ducat\AppData\Local\Temp\ipykernel_17820\2850162604.py:16: UserWarning: The given float value must not exceed 0.3902989066702947. But, you have given threshold=5.732571608525504.
  table = get_clusters_table(
C:\Users\ducat\AppData\Local\Temp\ipykernel_17820\2850162604.py:16: UserWarning: No clusters found with stat higher than 5.732571608525504
  table = get_clusters_table(
C:\Users\ducat\AppData\Local\Temp\ipykernel_17820\2850162604.py:16: UserWarning: The given float value must not exceed 1.6179829858740493. But, you have given threshold=5.003354962796608.
  table = get_clusters_table(
C:\Users

In [5]:
found_regions

{'high_over_low': {0.05: ['Lingual_L',
   'Precuneus_L',
   'Cerebellum_8_L',
   'Cerebellum_9_R',
   'Frontal_Sup_Medial_R',
   'Cerebellum_6_R',
   'Insula_R',
   'Insula_R'],
  0.01: ['Lingual_L',
   'Precuneus_L',
   'Cerebellum_8_L',
   'Cerebellum_9_R',
   'Frontal_Sup_Medial_R',
   'Cerebellum_6_R',
   'Insula_R',
   'Insula_R']},
 'undecided_over_high_+_low': {0.05: [], 0.01: []},
 '85_+_95_over_5_+_15': {0.05: ['Insula_L',
   'Temporal_Mid_L',
   'Frontal_Mid_2_L',
   'Putamen_L',
   'Frontal_Inf_Orb_2_L',
   'Frontal_Inf_Tri_L',
   'Insula_L',
   'Precuneus_L',
   'Cerebellum_9_R',
   'Cerebellum_8_R',
   'Insula_R',
   'Insula_R',
   'Temporal_Pole_Sup_R'],
  0.01: ['Insula_L',
   'Temporal_Mid_L',
   'Frontal_Mid_2_L',
   'Putamen_L',
   'Frontal_Inf_Orb_2_L',
   'Frontal_Inf_Tri_L',
   'Insula_L',
   'Precuneus_L',
   'Cerebellum_9_R',
   'Cerebellum_8_R',
   'Insula_R',
   'Insula_R',
   'Temporal_Pole_Sup_R']},
 '25_+_35_over_5_+_15': {0.05: [], 0.01: []},
 '55_+_65_over

In [6]:
from pprint import pprint

pprint(found_regions)

{'25_+_35_over_5_+_15': {0.01: [], 0.05: []},
 '35_+_45_over_5_+_15': {0.01: [], 0.05: []},
 '45_+_55_over_5_+_15': {0.01: ['Frontal_Sup_2_L',
                                'Frontal_Mid_2_L',
                                'Precuneus_L',
                                'LC_R',
                                'Cingulate_Mid_R',
                                'Thal_VA_R',
                                'Cerebellum_8_R',
                                'Hippocampus_R',
                                'Cerebellum_8_R',
                                'Temporal_Pole_Mid_R'],
                         0.05: ['Frontal_Sup_2_L',
                                'Frontal_Mid_2_L',
                                'Precuneus_L',
                                'LC_R',
                                'Cingulate_Mid_R',
                                'Thal_VA_R',
                                'Cerebellum_8_R',
                                'Hippocampus_R',
                                'C